# 01 — Data inventory

This notebook surveys every dataset used in the manuscript.

## Cohorts

| Cohort | n | Modality | Role |
|--------|---|----------|------|
| TCGA-THCA primary tumours | 513 | bulk RNA-seq + WES | **primary discovery cohort** for DM1/DM2 axis + 8-gene panel + TERT 4-group |
| GSE76039 PDTC + ATC | 37 | bulk RNA-seq | **external trajectory validation** — direction-correct AUC 0.974 |
| GSE27155 / GSE33630 / GSE29265 | 99+ | microarray | additional external transfer cohorts |
| GSE126698, GSE213647 | 100s | bulk | additional external transfer |
| GSE184362 scRNA | 66,000 cells (7 patients) | single-cell RNA-seq | hot/cold immune evidence |
| CCLE thyroid | 13 cell lines (5 vs 5 PRISM-covered) | bulk RNA-seq + drug screen | drug actionability mechanism-class |
| cBioPortal `thca_tcga_pub` | 36 of 504 patients | TERT promoter MAF | Liu–Xing 4-group survival (R8, Fig 8) |

## Files inspected below

1. `sample_master_v17_tert_v2.tsv` — primary cohort with TERT integrated, 513 rows × ~45 cols
2. `A2_dm_score_full_cohort.tsv` — DM1/DM2 cluster + driver assignment per sample (used by Fig 4, Fig 6)
3. `dark_matter_cluster_markers.tsv` — 41 cluster-defining genes (used by Fig 1, Fig 2)
4. `FINAL_extended_summary.json` — TERT recovery survival summary (used by Fig 8, R8)


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/home/seungho/personal/THCA_data_analysis/project/notebooks_or_scripts')
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 140)

from pathlib import Path
ROOT = Path('/home/seungho/personal/THCA_data_analysis/project/results')

## 1. Sample master with TERT integration (513 patients)

Primary backbone table. Each row = one TCGA-THCA primary tumour.
Key columns:
- `tert_promoter_integrated` — wildtype / mutated (recovered from cBioPortal Sanger-validated MAF)
- `quad_group` — A_braf_only / B_ras_only / D_triple_negative (TERT⁺ overlay creates the 4-group)
- `os_event`, `os_days` — overall survival (504 patients have follow-up; 16 events)
- `v17_dark_cluster` — DM1 / DM2 (179 of 513 have unsupervised cluster assignment)


In [ ]:
sm = pd.read_csv(ROOT / 'v17_tert_recovery' / 'v2' / 'sample_master_v17_tert_v2.tsv', sep='\t', low_memory=False)
print('rows:', len(sm))
print('TERT integrated:', sm['tert_promoter_integrated'].value_counts().to_dict())
print('quad_group:', sm['quad_group'].value_counts().to_dict())
print('OS events:', int(sm['os_event'].sum()), '/ ', sm['os_event'].notna().sum())
sm[['sample_id','quad_group','tert_promoter_integrated','os_event','os_days','v17_dark_cluster','tds16_score_v17']].head(10)

## 2. DM1/DM2 cluster + driver assignment (513 cohort with DM scores)

Each row has `prob_dm1`, `prob_dm2`, `dm_like` (final assignment), `driver_anchor`, `rai_score_recalc`.
This is the table that drives Fig 4 (driver × DM contingency) and Fig 6 (8-gene panel target).


In [ ]:
dm = pd.read_csv(ROOT / 'v17p3' / 'tables' / 'A2_dm_score_full_cohort.tsv', sep='\t', low_memory=False)
print('rows:', len(dm))
print('driver:', dm['driver_anchor'].value_counts().to_dict())
print('dm_like:', dm['dm_like'].value_counts().to_dict())
print('crosstab driver × dm_like:')
print(pd.crosstab(dm['driver_anchor'], dm['dm_like']))
dm[['sample_id','driver_anchor','dm_like','prob_dm1','prob_dm2','tds_score','rai_score_recalc','histology_subtype']].head(10)

## 3. Cluster-defining markers (41 genes, FDR < 1e-30)

Each row = one differentially-expressed gene with cluster assignment + log2FC + FDR.


In [ ]:
mk = pd.read_csv(ROOT / 'v17' / 'tables' / 'dark_matter_cluster_markers.tsv', sep='\t')
print('marker rows:', len(mk))
print('top DM1 markers (cluster=0):')
display(mk[mk['cluster']==0].sort_values('fdr').head(10))
print('top DM2 markers (cluster=1):')
display(mk[mk['cluster']==1].sort_values('fdr').head(10))

## 4. TERT recovery survival summary (used by Fig 8, R8)

Computed from cBioPortal `thca_tcga_pub` mirror (Sanger-validated TCGA Cell 2014 MAF).
**Honest reframe note**: while the univariate signal is strong (HR 6.31), it is largely captured by stage and age — multivariate HR drops to 1.88 (p = 0.29). See Fig 8 / robustness notebooks.


In [ ]:
import json
summary = json.loads((ROOT / 'v17_tert_recovery' / 'v2' / 'FINAL_extended_summary.json').read_text())
print(json.dumps(summary['survival'], indent=2))